# Tarefa 3 — CNNs (PyTorch) para o Multiprova Corretor (EMNIST)

Aluno: Lucas Medeiros — Disciplina: Aprendizado Profundo — PPgTI/IMD/UFRN

Reproducao, em PyTorch, das melhores arquiteturas de CNN do artigo Silva Filho et al. (2022)
para o app Multiprova Corretor, aplicadas a 3 subproblemas construidos a partir do EMNIST:
digitos (1-5), V ou F, letras (A-E).

**Antes de rodar:** em "Ambiente de execucao" -> "Alterar o tipo de ambiente de execucao",
selecione GPU (T4), para acelerar o treino.


## 1) Setup e download do EMNIST

In [1]:
pip install torch torchvision optuna tensorflow scikit-learn pandas

   ---------------------------------------- 0.0/124.1 MB ? eta -:--:--
   -- ------------------------------------- 8.7/124.1 MB 44.2 MB/s eta 0:00:03
   ----- ---------------------------------- 18.1/124.1 MB 46.1 MB/s eta 0:00:03
   --------- ------------------------------ 28.8/124.1 MB 47.1 MB/s eta 0:00:03
   ------------ --------------------------- 38.0/124.1 MB 46.8 MB/s eta 0:00:02
   --------------- ------------------------ 47.4/124.1 MB 46.6 MB/s eta 0:00:02
   ------------------ --------------------- 56.6/124.1 MB 46.0 MB/s eta 0:00:02
   --------------------- ------------------ 66.3/124.1 MB 45.7 MB/s eta 0:00:02
   ------------------------ --------------- 75.5/124.1 MB 45.3 MB/s eta 0:00:02
   --------------------------- ------------ 84.1/124.1 MB 44.8 MB/s eta 0:00:01
   ------------------------------ --------- 93.6/124.1 MB 44.9 MB/s eta 0:00:01
   -------------------------------- ------ 102.2/124.1 MB 44.6 MB/s eta 0:00:01
   ---------------------------------- ---- 111.1/1

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import torchvision
from torchvision import transforms
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
import os, time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
])

print("Baixando EMNIST (digits)...")
emnist_digits_train = torchvision.datasets.EMNIST(root="./data", split="digits", train=True, download=True, transform=transform)
emnist_digits_test = torchvision.datasets.EMNIST(root="./data", split="digits", train=False, download=True, transform=transform)

print("Baixando EMNIST (letters)...")
emnist_letters_train = torchvision.datasets.EMNIST(root="./data", split="letters", train=True, download=True, transform=transform)
emnist_letters_test = torchvision.datasets.EMNIST(root="./data", split="letters", train=False, download=True, transform=transform)

print("digits:", len(emnist_digits_train), len(emnist_digits_test))
print("letters:", len(emnist_letters_train), len(emnist_letters_test))


Device: cpu
Baixando EMNIST (digits)...


100%|██████████| 562M/562M [00:14<00:00, 39.9MB/s] 


Baixando EMNIST (letters)...
digits: 240000 40000
letters: 124800 20800


## 2) Arquitetura CNN e funcoes de treino/avaliacao

In [4]:
class CNNMultiprova(nn.Module):
    def __init__(self, n_classes, in_channels=1):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 2 * 2, n_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


def make_filtered_subset(dataset, label_map):
    idx = [i for i, (_, y) in enumerate(dataset) if int(y) in label_map]
    sub = Subset(dataset, idx)
    return sub


class RemappedDataset(torch.utils.data.Dataset):
    def __init__(self, base, label_map):
        self.base = base
        self.label_map = label_map

    def __len__(self):
        return len(self.base)

    def __getitem__(self, i):
        x, y = self.base[i]
        return x, self.label_map[int(y)]


def build_problem_datasets(train_ds, test_ds, label_map, batch_size=128, val_frac=0.15):
    idx_train_all = [i for i in range(len(train_ds)) if int(train_ds[i][1]) in label_map]
    idx_test = [i for i in range(len(test_ds)) if int(test_ds[i][1]) in label_map]

    n_val = int(len(idx_train_all) * val_frac)
    rng = np.random.RandomState(42)
    idx_train_all = np.array(idx_train_all)
    rng.shuffle(idx_train_all)
    idx_val = idx_train_all[:n_val].tolist()
    idx_train = idx_train_all[n_val:].tolist()

    train_sub = RemappedDataset(Subset(train_ds, idx_train), label_map)
    val_sub = RemappedDataset(Subset(train_ds, idx_val), label_map)
    test_sub = RemappedDataset(Subset(test_ds, idx_test), label_map)

    train_loader = DataLoader(train_sub, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_sub, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_sub, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader, test_loader, len(idx_train), len(idx_val), len(idx_test)


def train_model(model, train_loader, val_loader, epochs=12, lr=1e-3, patience=3):
    model.to(device)
    opt = optim.Adam(model.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss()
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_val = float("inf")
    bad_epochs = 0
    for ep in range(epochs):
        model.train()
        tot, correct, loss_sum = 0, 0, 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            out = model(xb)
            loss = crit(out, yb)
            loss.backward()
            opt.step()
            loss_sum += loss.item() * xb.size(0)
            correct += (out.argmax(1) == yb).sum().item()
            tot += xb.size(0)
        train_loss = loss_sum / tot
        train_acc = correct / tot

        model.eval()
        tot, correct, loss_sum = 0, 0, 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                out = model(xb)
                loss = crit(out, yb)
                loss_sum += loss.item() * xb.size(0)
                correct += (out.argmax(1) == yb).sum().item()
                tot += xb.size(0)
        val_loss = loss_sum / tot
        val_acc = correct / tot

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        print(f"epoch {ep+1}/{epochs}  train_loss={train_loss:.4f} train_acc={train_acc:.4f}  val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

        if val_loss < best_val - 1e-4:
            best_val = val_loss
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                print("Early stopping.")
                break
    return history


def evaluate(model, loader, class_names):
    model.eval()
    all_preds, all_true = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            out = model(xb)
            preds = out.argmax(1).cpu().numpy()
            all_preds.extend(preds.tolist())
            all_true.extend(yb.numpy().tolist())
    acc = np.mean(np.array(all_preds) == np.array(all_true))
    print("Acuracia teste:", acc)
    print(classification_report(all_true, all_preds, target_names=class_names))
    cm = confusion_matrix(all_true, all_preds)
    return acc, cm, all_true, all_preds


## 3) Treino e avaliacao dos 3 subproblemas

In [5]:
results = {}

# ---------- Problema 1: digitos 1-5 ----------
digits_map = {1: 0, 2: 1, 3: 2, 4: 3, 5: 4}
digits_classes = ["1", "2", "3", "4", "5"]
tl, vl, tel, ntr, nva, nte = build_problem_datasets(emnist_digits_train, emnist_digits_test, digits_map)
print(f"[digitos] treino={ntr} val={nva} teste={nte}")
model_digits = CNNMultiprova(n_classes=len(digits_classes))
hist_digits = train_model(model_digits, tl, vl, epochs=12)
acc_digits, cm_digits, yt_d, yp_d = evaluate(model_digits, tel, digits_classes)
results["digitos"] = {"acc": acc_digits, "history": hist_digits, "cm": cm_digits}

# ---------- Problema 2: V ou F ----------
vf_map = {22: 0, 6: 1}  # v=22, f=6 (emnist 'letters': 1=a ... 26=z)
vf_classes = ["V", "F"]
tl2, vl2, tel2, ntr2, nva2, nte2 = build_problem_datasets(emnist_letters_train, emnist_letters_test, vf_map)
print(f"[V ou F] treino={ntr2} val={nva2} teste={nte2}")
model_vf = CNNMultiprova(n_classes=len(vf_classes))
hist_vf = train_model(model_vf, tl2, vl2, epochs=12)
acc_vf, cm_vf, yt_vf, yp_vf = evaluate(model_vf, tel2, vf_classes)
results["vf"] = {"acc": acc_vf, "history": hist_vf, "cm": cm_vf}

# ---------- Problema 3: letras A-E ----------
letters_map = {1: 0, 2: 1, 3: 2, 4: 3, 5: 4}  # a..e
letters_classes = ["A", "B", "C", "D", "E"]
tl3, vl3, tel3, ntr3, nva3, nte3 = build_problem_datasets(emnist_letters_train, emnist_letters_test, letters_map)
print(f"[letras A-E] treino={ntr3} val={nva3} teste={nte3}")
model_letters = CNNMultiprova(n_classes=len(letters_classes))
hist_letters = train_model(model_letters, tl3, vl3, epochs=12)
acc_letters, cm_letters, yt_l, yp_l = evaluate(model_letters, tel3, letters_classes)
results["letras"] = {"acc": acc_letters, "history": hist_letters, "cm": cm_letters}

print("\nRESUMO DE ACURACIA (teste):")
for k, v in results.items():
    print(f"  {k}: {v['acc']:.4f}")


[digitos] treino=102000 val=18000 teste=20000
epoch 1/12  train_loss=0.0744 train_acc=0.9752  val_loss=0.0197 val_acc=0.9940
epoch 2/12  train_loss=0.0156 train_acc=0.9953  val_loss=0.0149 val_acc=0.9964
epoch 3/12  train_loss=0.0099 train_acc=0.9971  val_loss=0.0137 val_acc=0.9961
epoch 4/12  train_loss=0.0086 train_acc=0.9976  val_loss=0.0097 val_acc=0.9976
epoch 5/12  train_loss=0.0068 train_acc=0.9980  val_loss=0.0132 val_acc=0.9966
epoch 6/12  train_loss=0.0063 train_acc=0.9982  val_loss=0.0108 val_acc=0.9971
epoch 7/12  train_loss=0.0052 train_acc=0.9983  val_loss=0.0109 val_acc=0.9978
Early stopping.
Acuracia teste: 0.9975
              precision    recall  f1-score   support

           1       1.00      1.00      1.00      4000
           2       1.00      1.00      1.00      4000
           3       1.00      1.00      1.00      4000
           4       1.00      1.00      1.00      4000
           5       1.00      1.00      1.00      4000

    accuracy                        

## 4) Exportacao dos modelos e tabela de parametros/tamanho (item 3.1)

In [6]:
import pandas as pd

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

os.makedirs("modelos", exist_ok=True)
torch.save(model_digits.state_dict(), "modelos/cnn_digitos.pt")
torch.save(model_vf.state_dict(), "modelos/cnn_vf.pt")
torch.save(model_letters.state_dict(), "modelos/cnn_letras.pt")

rows = []
for name, model, path in [
    ("Digitos (1-5)", model_digits, "modelos/cnn_digitos.pt"),
    ("V ou F", model_vf, "modelos/cnn_vf.pt"),
    ("Letras (A-E)", model_letters, "modelos/cnn_letras.pt"),
]:
    n_params = count_params(model)
    size_mb = os.path.getsize(path) / (1024 * 1024)
    rows.append({"Modelo": name, "Parametros treinaveis": n_params, "Tamanho (MB)": round(size_mb, 3)})

df_models = pd.DataFrame(rows)
print(df_models.to_string(index=False))
df_models.to_csv("modelos/resumo_modelos.csv", index=False)


       Modelo  Parametros treinaveis  Tamanho (MB)
Digitos (1-5)                  93957         0.363
       V ou F                  93186         0.359
 Letras (A-E)                  93957         0.363
